<a href="https://colab.research.google.com/github/Marcin19721205/ProcessControl/blob/main/CloseLoopTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import math  # funkcje matematyczne

def get_float(prompt: str) -> float:
    while True:  # pytaj aż użytkownik poda poprawną liczbę
        try:
            return float(input(prompt).replace(",", "."))  # akceptuj przecinek i kropkę
        except ValueError:
            print("Błąd: podaj poprawną liczbę.")  # walidacja wejścia

def simc_closed_loop_from_p(tp: float, dyp: float, dyu: float, Kc0: float, dys: float) -> dict:
    if Kc0 == 0:  # bez tego poleci dzielenie przez zero
        raise ValueError("Kc0 nie może być równe 0.")
    if tp <= 0:  # czas piku musi być dodatni
        raise ValueError("tp musi być > 0.")

    y_inf = 0.45 * (dyp + dyu)  # estymata zmiany ustalonej y∞
    if abs(y_inf) < 1e-12:  # ochrona przed dzieleniem przez zero
        raise ValueError("Wyznaczone y_inf jest równe 0 lub zbyt bliskie 0.")

    D = (dyp - y_inf) / y_inf  # względne przeregulowanie
    B = abs((dys - y_inf) / y_inf)  # bezwzględny offset ustalony
    if B < 1e-12:  # bez tego k i r nie istnieją
        raise ValueError("Wyznaczone B jest równe 0 lub zbyt bliskie 0.")

    A = 1.152 * D**2 - 1.607 * D + 1.0  # współczynnik korelacji SIMC
    r = 2.0 * A / B  # r = tau1/theta
    theta = tp * (0.309 + 0.209 * math.exp(-0.61 * r))  # dead time
    tau1 = r * theta  # dominująca stała czasowa
    k = 1.0 / (Kc0 * B)  # process gain

    return {
        "y_inf": y_inf,
        "D": D,
        "B": B,
        "A": A,
        "r": r,
        "k_process_gain": k,
        "Theta_dead_time": theta,
        "tau1_time_constant": tau1,
    }

tp = get_float("Podaj tp - czas max pierwszego piku oscylacji: ")  # czas do 1. maksimum
dyp = get_float("Podaj dyp - wartość max pierwszej oscylacji: ")  # yp
dyu = get_float("Podaj dyu - min drugiej oscylacji (pierwszy undershoot): ")  # yu
Kc0 = get_float("Podaj Kc0 - P-controller gain: ")  # wzmocnienie regulatora P
dys = get_float("Podaj dys - SP change: ")  # skok SP

try:
    res = simc_closed_loop_from_p(tp, dyp, dyu, Kc0, dys)  # obliczenia główne

    print("\n" + "-" * 50)
    print("WYNIKI SIMC - CLOSED LOOP TEST (P-only)")
    print("-" * 50)
    print(f"y_inf              = {res['y_inf']:.6f}")
    print(f"D                  = {res['D']:.6f}")
    print(f"B                  = {res['B']:.6f}")
    print(f"A                  = {res['A']:.6f}")
    print(f"r = tau1/theta     = {res['r']:.6f}")
    print("Process Parameters")
    print(f"k_process_gain     = {res['k_process_gain']:.6f}")
    print(f"Theta_dead_time    = {res['Theta_dead_time']:.6f}")
    print(f"tau1_time_constant = {res['tau1_time_constant']:.6f}")
    print("-" * 50)
except ValueError as e:
    print(f"\nBłąd obliczeń: {e}")

Podaj tp - czas max pierwszego piku oscylacji: 8.6
Podaj dyp - wartość max pierwszej oscylacji: 1.15
Podaj dyu - min drugiej oscylacji (pierwszy undershoot): 0.65
Podaj Kc0 - P-controller gain: 6
Podaj dys - SP change: 1

--------------------------------------------------
WYNIKI SIMC - CLOSED LOOP TEST (P-only)
--------------------------------------------------
y_inf              = 0.810000
D                  = 0.419753
B                  = 0.234568
A                  = 0.528431
r = tau1/theta     = 4.505567
Process Parameters
k_process_gain     = 0.710526
Theta_dead_time    = 2.772488
tau1_time_constant = 12.491633
--------------------------------------------------
